In [0]:
from common_utils.ingester import read_s3_parquet,write_raw
from pymongo import MongoClient
from common_utils.logging import get_logger 
from datetime import date
import json

# ======================================
# Logger set up
# ======================================
logger = get_logger('s3-ingestion')

# ======================================
# Config Path
# ======================================
dbutils.widgets.text("path","")
config_path=dbutils.widgets.get('path')
print(config_path)

# ======================================
# Read Config
# ======================================
with open(config_path) as f:
    config=json.load(f)

# Configuration
config_source=config['source']
config_target=config['target']

run_date=date.today().isoformat()

aws_access_key_id=dbutils.secrets.get(scope='retail-platform-dev',key=config_source['aws_access_key_id'])
aws_secret_access_key=dbutils.secrets.get(scope='retail-platform-dev',key=config_source['aws_secret_access_key'])
source_path = (
    f"s3a://{aws_access_key_id}:{aws_secret_access_key}"
    f"@bucket-rivadata/retail/sales.parquet"
)

# ======================================
# Reading from AWS Database
# ======================================
df=read_s3_parquet(spark,aws_access_key_id, aws_secret_access_key, "bucket-rivadata/retail/sales.parquet")

# ======================================
# Writing data to store at our local machine
# ======================================
target_path = f'{config_target["base_path"]}{config_target["folder"]}/load_date={run_date}'
logger.info('Data stored at location %s',target_path)
write_raw(df, target_path,config_target['file_format'], config_target['mode'], options = None)


In [0]:
from datetime import date
run_date=date.today().isoformat()
target_path = f'/Volumes/retaildataplatform/bronze/raw_data/sales/load_date={run_date}'


In [0]:
# Another way to read the data from s3 using dbutils.filessystem.copy
# dbutils.fs.cp(source_path,target_path)

In [0]:
df=spark.read.format('parquet').load(source_path)
df.display()

In [0]:
df.write.format('parquet').mode('overwrite').save(target_path)
df.display()